In [46]:
import pandas as pd
from dotenv import load_dotenv
import re
import time

import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
import random



load_dotenv()

True

In [47]:
df = pd.read_csv("trump_tweets.csv")
df.head()

,url,username,DateKey,TimeKey,Text,deleted
0,https://trumpstruth.org/statuses/37674,realDonaldTrump,20260409,163400,NaN,0
1,https://trumpstruth.org/statuses/37673,realDonaldTrump,20260409,144000,"None of these people, including our own, very ...",0
2,https://trumpstruth.org/statuses/37672,realDonaldTrump,20260409,54600,"All U.S. Ships, Aircraft, and Military Personn...",0
3,https://trumpstruth.org/statuses/37671,realDonaldTrump,20260409,45600,The Failing New York Times and Fake News CNN e...,0
4,https://trumpstruth.org/statuses/37670,realDonaldTrump,20260409,13100,"NATO WASN’T THERE WHEN WE NEEDED THEM, AND THE...",0


In [48]:
def create_driver(headless = False):
    options = uc.ChromeOptions()
    if headless:
        options.add_argument('--headless')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--window-size=1920,1080')
    options.add_argument('--disable-blink-features=AutomationControlled')
    return uc.Chrome(version_main=146, options=options)

In [49]:
driver = create_driver()

In [50]:
def get_original_truth_url(driver):
    try:
        # 1. Wacht tot het element aanwezig is (max 10 seconden)
        # We zoeken naar de 'a' tag die binnen de specifieke class 'status-details-table__value' valt
        wait = WebDriverWait(driver, 10)
        link_element = wait.until(EC.presence_of_element_located(
            (By.CSS_SELECTOR, "td.status-details-table__value a")
        ))
        
        # 2. Haal het href attribuut op
        original_url = link_element.get_attribute("href")
        return original_url
        
    except Exception as e:
        print(f"⚠️ Kon de originele URL niet vinden: {e}")
        return None

In [51]:
for index, url in enumerate(df['url']):
    driver.get(url=url)
    df.at[index,'original_url'] = get_original_truth_url(driver=driver)    
    

In [52]:
print(df.head())

                                      url         username   DateKey  TimeKey  \
0  https://trumpstruth.org/statuses/37674  realDonaldTrump  20260409   163400   
1  https://trumpstruth.org/statuses/37673  realDonaldTrump  20260409   144000   
2  https://trumpstruth.org/statuses/37672  realDonaldTrump  20260409    54600   
3  https://trumpstruth.org/statuses/37671  realDonaldTrump  20260409    45600   
4  https://trumpstruth.org/statuses/37670  realDonaldTrump  20260409    13100   

                                                Text  deleted  \
0                                                NaN        0   
1  None of these people, including our own, very ...        0   
2  All U.S. Ships, Aircraft, and Military Personn...        0   
3  The Failing New York Times and Fake News CNN e...        0   
4  NATO WASN’T THERE WHEN WE NEEDED THEM, AND THE...        0   

                                        original_url  
0  https://truthsocial.com/@realDonaldTrump/11637...  
1  https://t

In [53]:
def bypass_cloudflare(driver):
    """Probeert door de Cloudflare 'Turnstile' muur te breken."""
    try:
        time.sleep(5)
        if "Cloudflare" in driver.title or "Just a moment" in driver.title:
            time.sleep(2)
            print("🛡️ Cloudflare gedetecteerd. Poging tot interactie...")
            iframes = driver.find_elements(By.TAG_NAME, "iframe")
            for index, iframe in enumerate(iframes):
                try:
                    actions = ActionChains(driver)
                    actions.move_to_element(iframe).pause(random.uniform(0.1, 0.5)).click().perform()
                    print(f"✅ Geklikt op verificatie-element {index}")
                    time.sleep(7)
                    break
                except:
                    continue
    except Exception as e:
        print(f"⚠️ Cloudflare bypass mislukt: {e}")

In [54]:
def getMetaData(driver):
    """
    Haalt likes, reposts (retruths) en comments (replies) op.
    Inclusief Cloudflare bypass en 'K' naar getal conversie.
    """
    
    # 1. Eerst de Cloudflare check
    bypass_cloudflare(driver)
    
    # 2. Helper functie voor de getal-conversie (8.9k -> 8900)
    def parse_truth_number(text):
        if not text: return 0
        # Verwijder woorden, behoud cijfers, punten en de 'k'
        clean = text.lower().replace('replies', '').replace('likes', '').replace('retruths', '').strip()
        multiplier = 1000 if 'k' in clean else 1
        try:
            # Haal alleen het getal deel op (bijv 8.9 uit 8.9k)
            num_part = re.sub(r'[^\d.]', '', clean.replace('k', ''))
            return int(float(num_part) * multiplier)
        except:
            return 0

    # 3. Wacht even tot de elementen geladen zijn (lazy loading)
    try:
        wait = WebDriverWait(driver, 10)
        
        # Wacht tot tenminste één van de stats zichtbaar is
        wait.until(EC.presence_of_element_located((By.XPATH, "//*[contains(text(), 'Likes') or contains(text(), 'replies')]")))
        
        # --- Extractie ---
        # Likes
        try:
            likes_text = driver.find_element(By.XPATH, "//div[text()='Likes']/preceding-sibling::p").text
            likes = parse_truth_number(likes_text)
        except: likes = 0

        # Reposts (ReTruths)
        try:
            reposts_text = driver.find_element(By.XPATH, "//div[text()='ReTruths']/preceding-sibling::p").text
            reposts = parse_truth_number(reposts_text)
        except: reposts = 0

        # Comments (Replies)
        try:
            # Zoek de <p> die 'replies' bevat
            comments_text = driver.find_element(By.XPATH, "//p[contains(text(), 'replies')]").text
            comments = parse_truth_number(comments_text)
        except: comments = 0

        return likes, reposts, comments

    except Exception as e:
        print(f"⚠️ Metadata extractie onderbroken: {e}")
        return 0, 0, 0

In [55]:
df.head()

,url,username,DateKey,TimeKey,Text,deleted,original_url
0,https://trumpstruth.org/statuses/37674,realDonaldTrump,20260409,163400,NaN,0,https://truthsocial.com/@realDonaldTrump/11637...
1,https://trumpstruth.org/statuses/37673,realDonaldTrump,20260409,144000,"None of these people, including our own, very ...",0,https://truthsocial.com/@realDonaldTrump/11637...
2,https://trumpstruth.org/statuses/37672,realDonaldTrump,20260409,54600,"All U.S. Ships, Aircraft, and Military Personn...",0,https://truthsocial.com/@realDonaldTrump/11637...
3,https://trumpstruth.org/statuses/37671,realDonaldTrump,20260409,45600,The Failing New York Times and Fake News CNN e...,0,https://truthsocial.com/@realDonaldTrump/11637...
4,https://trumpstruth.org/statuses/37670,realDonaldTrump,20260409,13100,"NATO WASN’T THERE WHEN WE NEEDED THEM, AND THE...",0,https://truthsocial.com/@realDonaldTrump/11637...


In [56]:
# Gebruik iterrows om de ECHTE index en de rij-inhoud te krijgen
for index, row in df.iterrows():
    url = row['original_url']
    print(url)
    try:
        driver.get(url)
        scroll_randomly(driver=driver)
        # De wachttijd zit al IN getMetaData, dus hier hoeft geen sleep
        likes, reposts, comments = getMetaData(driver=driver)
            
        # Gebruik .at met de echte index van de rij
        df.at[index, 'likes'] = likes
        df.at[index, 'reposts'] = reposts
        df.at[index, 'comments'] = comments
        
    except Exception as e:
        print(f"❌ Fout bij index {index}: {e}")

https://truthsocial.com/@realDonaldTrump/116375240064230100
https://truthsocial.com/@realDonaldTrump/116374792489555954
https://truthsocial.com/@realDonaldTrump/116372694697146221
https://truthsocial.com/@realDonaldTrump/116372497116210545
https://truthsocial.com/@realDonaldTrump/116371693008302124
https://truthsocial.com/@realDonaldTrump/116369995519355709
https://truthsocial.com/@realDonaldTrump/116369934305888462
https://truthsocial.com/@realDonaldTrump/116369645448805093
https://truthsocial.com/@realDonaldTrump/116368854361048135
https://truthsocial.com/@realDonaldTrump/116368825638596650
https://truthsocial.com/@realDonaldTrump/116368738767645698
https://truthsocial.com/@realDonaldTrump/116367088879643074
https://truthsocial.com/@realDonaldTrump/116367011501074973
https://truthsocial.com/@realDonaldTrump/116367005523554168
https://truthsocial.com/@realDonaldTrump/116366146721038653
https://truthsocial.com/@realDonaldTrump/116366072136989268
https://truthsocial.com/@realDonaldTrump

KeyboardInterrupt: 

In [ ]:
driver.close()

InvalidSessionIdException: Message: invalid session id; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#invalidsessionidexception
Stacktrace:
	undetected_chromedriver!GetHandleVerifier [0x99cdf3+10b03]
	undetected_chromedriver!GetHandleVerifier [0x99cf24+10c34]
	undetected_chromedriver!(No symbol) [0x781f5e]
	undetected_chromedriver!(No symbol) [0x7bf4b7]
	undetected_chromedriver!(No symbol) [0x7ed9c6]
	undetected_chromedriver!(No symbol) [0x7e8e42]
	undetected_chromedriver!(No symbol) [0x7e8462]
	undetected_chromedriver!(No symbol) [0x75534e]
	undetected_chromedriver!(No symbol) [0x7558ee]
	undetected_chromedriver!(No symbol) [0x755dcd]
	undetected_chromedriver!GetHandleVerifier [0xc048b9+2785c9]
	undetected_chromedriver!GetHandleVerifier [0xbffeb5+273bc5]
	undetected_chromedriver!GetHandleVerifier [0xc1e06b+291d7b]
	undetected_chromedriver!GetHandleVerifier [0x9b5cc8+299d8]
	undetected_chromedriver!GetHandleVerifi

In [ ]:
df.head(10)

,url,username,DateKey,TimeKey,Text,deleted,original_url,likes,reposts,comments
0,https://trumpstruth.org/statuses/37672,realDonaldTrump,20260409,54600,"All U.S. Ships, Aircraft, and Military Personn...",0,https://truthsocial.com/@realDonaldTrump/11637...,30900.0,7560.0,4504.0
1,https://trumpstruth.org/statuses/37671,realDonaldTrump,20260409,45600,The Failing New York Times and Fake News CNN e...,0,https://truthsocial.com/@realDonaldTrump/11637...,24200.0,6260.0,2192.0
2,https://trumpstruth.org/statuses/37670,realDonaldTrump,20260409,13100,"NATO WASN’T THERE WHEN WE NEEDED THEM, AND THE...",0,https://truthsocial.com/@realDonaldTrump/11637...,53600.0,11800.0,7866.0
3,https://trumpstruth.org/statuses/37669,realDonaldTrump,20260408,182000,Marjorie “Traitor” Brown’s (GREEN TURNS TO BRO...,0,https://truthsocial.com/@realDonaldTrump/11636...,36600.0,7630.0,4016.0
4,https://trumpstruth.org/statuses/37668,realDonaldTrump,20260408,180400,"Numerous Agreements, Lists, and Letters are be...",0,https://truthsocial.com/@realDonaldTrump/11636...,45300.0,10800.0,4861.0
5,https://trumpstruth.org/statuses/37667,realDonaldTrump,20260408,165100,Secretary Hegseth and Chairman Caine hold a pr...,0,https://truthsocial.com/@realDonaldTrump/11636...,18500.0,4280.0,1497.0
6,https://trumpstruth.org/statuses/37666,realDonaldTrump,20260408,133000,A Country supplying Military Weapons to Iran w...,0,https://truthsocial.com/@realDonaldTrump/11636...,41300.0,9280.0,3932.0
7,https://trumpstruth.org/statuses/37665,realDonaldTrump,20260408,132200,"The United States will work closely with Iran,...",0,https://truthsocial.com/@realDonaldTrump/11636...,32700.0,7620.0,3810.0
8,https://trumpstruth.org/statuses/37664,realDonaldTrump,20260408,130000,NaN,0,https://truthsocial.com/@realDonaldTrump/11636...,15600.0,3450.0,1769.0
9,https://trumpstruth.org/statuses/37663,realDonaldTrump,20260408,60100,A big day for World Peace! Iran wants it to ha...,0,https://truthsocial.com/@realDonaldTrump/11636...,38400.0,8930.0,6901.0
